<a href="https://colab.research.google.com/github/nmwaura4/Projects/blob/main/Copy_of_monty_Analysis_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score,root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
%matplotlib inline

In [ ]:
df=pd.read_excel('/content/Monty_3.xlsx')
df.head()

In [ ]:
df.shape

In [ ]:
list(df.columns)

In [ ]:
X_df=df.drop(columns=['Unnamed: 19','Unnamed: 20','Unnamed: 21','Unnamed: 22'])

In [ ]:
X_df

In [ ]:
X_df=pd.get_dummies(X_df,drop_first=True)
X_df.head()

In [ ]:
X_df.isna().sum()

In [ ]:
for col in X_df.columns:
  if X_df[col].dtypes=='bool':
    X_df[col]=X_df[col].astype(int)

In [ ]:
X_df=pd.get_dummies(X_df,drop_first=True)
X_df.head()

In [ ]:
X_df.sample(7)

In [ ]:
corr=X_df.corr()
corr

To create a more readable correlation plot, I will select the top 15 most important features based on the Random Forest model's feature importances. Then, I'll calculate and visualize the correlation matrix for these selected features.

In [ ]:
for col in X_df.columns:
  if X_df[col].dtypes=='object':
    X_df[col]=X_df[col].fillna(X_df[col].mode()[0])
  else:
    X_df[col]=X_df[col].fillna(X_df[col].mean())

In [ ]:
X_df.isna().sum()

In [ ]:
X=X_df.drop(columns=['montdorensis count (/g)'])
y=X_df['montdorensis count (/g)']

In [ ]:
model=RandomForestRegressor()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model=RandomForestRegressor()
model.fit(X_train,y_train)

In [ ]:
importance_df=pd.DataFrame(X_df.columns,columns=['Features'])
importance_df

In [ ]:
importance_df=pd.DataFrame(X.columns,columns=['Features'])
importance_df['Importance']=model.feature_importances_
importance_df=importance_df.sort_values(by='Importance',ascending=False)
importance_df

In [ ]:
top_15_features = importance_df.head(15)['Features'].tolist()

features_for_corr_heatmap = top_15_features + ['montdorensis count (/g)']

corr = X_df[features_for_corr_heatmap].corr()
corr

In [ ]:
plt.figure(figsize=(20, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Selected Features')
plt.show()

In [ ]:
model = RandomForestRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
Dataframe = ({'Actual': y_test, 'Predicted': y_pred})
new_df = pd.DataFrame(Dataframe)
display(new_df)

In [ ]:
# Calculate the difference between Actual and Predicted values
new_df['Difference'] = new_df['Actual'] - new_df['Predicted']

# Calculate the absolute difference
new_df['Absolute Difference'] = abs(new_df['Actual'] - new_df['Predicted'])

# Display the DataFrame with the new columns
display(new_df.head())

# Evaluation Metrics

In [ ]:
print(f'Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f}')
print(f'Mean Squared Error: {mean_squared_error(y_test, y_pred):.2f}')
print(f'Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')
print(f'R2 Score: {r2_score(y_test, y_pred):.2f}')

In [ ]:
future_dates=pd.DataFrame({'Date':pd.date_range(start='2026-01-01',end='2026-04-30',freq='D')})

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

num_predictions = len(new_df)
start_date = pd.to_datetime('2026-01-01')
dates_for_predictions = pd.Series(pd.date_range(start=start_date, periods=num_predictions, freq='D'), index=new_df.index)

# Scatter plot
plt.figure(figsize=(10,6))
plt.scatter(dates_for_predictions, new_df['Predicted'], label='Predictions')

# Convert dates to numbers for regression
x_numeric = pd.to_numeric(dates_for_predictions).values.reshape(-1,1)

# Target variable for trend line
y_trend = new_df['Predicted'].values

# Train model for trend line
trend_model = LinearRegression()
trend_model.fit(x_numeric, y_trend)

# Regression line
y_line = trend_model.predict(x_numeric)

# Plot line
plt.plot(dates_for_predictions, y_line, color='blue', label='Trend Line')

# Horizontal line (using the mean of predicted values as a baseline)
plt.axhline(y=new_df['Predicted'].mean(), color='red', linestyle='--', label='Mean Prediction')

plt.legend()
plt.title('Predicted Values and Trend over Time for Test Set')
plt.xlabel('Date')
plt.ylabel('Predicted montdorensis count (/g)')
plt.show()

### Future Prediction for 'montdorensis count (/g)'

To make future predictions, we need to create a dataset for future dates with relevant features. For simplicity, we'll assume the future feature values are the mean of their respective columns from the training data. This will provide a general trend prediction without specific assumptions about future changes in individual features.

Future Trends: The plot of predicted values for 'montdorensis count (/g)' over future dates shows an upward trend. This indicates that, based on the historical data and the features used, the model predicts an increase in 'montdorensis count (/g)' over the projected period.




### Residual Analysis

Residual analysis is crucial for evaluating a regression model's performance. It helps us understand if the model is making systematic errors and if its assumptions are met.

In [ ]:
# Calculate the difference between Actual and Predicted values (re-added)
new_df['Difference'] = new_df['Actual'] - new_df['Predicted']

plt.figure(figsize=(12, 5))

# Scatter plot of Predicted vs. Residuals
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.scatterplot(x=new_df['Predicted'], y=new_df['Difference'])
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Residuals vs. Predicted Values')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')

# Histogram of Residuals
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.histplot(new_df['Difference'], kde=True)
plt.title('Distribution of Residuals')
plt.xlabel('Residuals')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()





#### Interpretation of Residual Plots:

*   **Residuals vs. Predicted Values Plot:** Ideally, the residuals should be randomly scattered around zero, with no discernible pattern. If there's a pattern (e.g., a cone shape, a curve), it indicates that the model might not be capturing some underlying relationship in the data.
*   **Distribution of Residuals Plot:** For a good regression model, residuals should ideally be normally distributed around zero. This plot helps to visually check for normality.

Distribution of Residuals Plot: This histogram shows the frequency of different residual values. For a well-performing regression model, we ideally want the residuals to be normally distributed around zero. A bell-shaped curve centered at zero suggests that the model's errors are random and unbiased. Deviations from normality (e.g., skewed distributions or multiple peaks) could point to issues such as outliers, omitted variables, or an incorrect model specification. Given our low MAE, we would expect this distribution to be roughly centered around zero and appear somewhat normal.

In [ ]:
df.columns

In [ ]:
list(df.columns)

In [ ]:
df_selected=['Moisture content montdorensis counted','montdorensis count (/g)',
 'Ratio count',
 'carpo eggs/Tm mobiles count',
 'moisture content carpo',
 'carpo count (/g)',
 'carpo eggs count (/g)',
 'montdorensis/g at start up',
 'Ratio at start up',
 'carpo eggs/Tm at start up',
 '% eggs Tm','Average RH room',
 'Average temp (C)',]

In [ ]:
df_selected=df[df_selected]
df_selected.head()

In [ ]:
df_selected.isna().sum()

In [ ]:
df_selected.shape

In [ ]:
df_selected.fillna("")

In [ ]:
df_selected.isna().sum()

In [ ]:
for col in df_selected.columns:
  if df_selected[col].dtypes=='object':
    df_selected.loc[:, col]=df_selected[col].fillna(df_selected[col].mode()[0])
  else:
    df_selected.loc[:, col]=df_selected[col].fillna(df_selected[col].mean())

In [ ]:
df_selected.isna().sum()

In [ ]:
df_selected.drop(columns=['Moisture content montdorensis counted'],inplace=True)

### Converting Float Numbers to Whole Numbers

There are several ways to convert float numbers to whole numbers, each with slightly different behavior:

1.  **Using `astype(int)`:** This truncates the decimal part, effectively performing a floor operation for positive numbers and a ceiling for negative numbers (e.g., 3.7 -> 3, -3.7 -> -3).
2.  **Using `round()`:** This rounds to the nearest integer. By default, `round()` rounds to the nearest even number when a number is exactly halfway between two integers (e.g., 2.5 -> 2, 3.5 -> 4).
3.  **Using `np.floor()`:** This rounds down to the nearest integer (e.g., 3.7 -> 3, -3.7 -> -4).
4.  **Using `np.ceil()`:** This rounds up to the nearest integer (e.g., 3.7 -> 4, -3.7 -> -3).

In [ ]:
# Convert all columns to numeric, coercing errors to NaN
for col in df_selected.columns:
    df_selected.loc[:, col] = pd.to_numeric(df_selected[col], errors='coerce')

# Fill any newly created NaNs (from 'N.A.' or other non-numeric strings) with the mean of the column
for col in df_selected.columns:
    if df_selected[col].isna().any(): # Check if there are any NaNs in the column
        df_selected.loc[:, col] = df_selected[col].fillna(df_selected[col].mean())

corr = df_selected.corr()
corr

In [ ]:
plt.figure(figsize=(20, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Selected Features')
plt.show()

In [ ]:
importance_df=pd.DataFrame(X.columns,columns=['Features'])
importance_df['Importance']=model.feature_importances_
importance_df=importance_df.sort_values(by='Importance',ascending=False)
importance_df